# Semantic Search Over GitHub Issues with Oracle AI Vector Search

This notebook pulls real issues from a public GitHub repo, stores them as vector embeddings in Oracle AI Database 26ai using `langchain-oracledb`, and runs semantic similarity queries with metadata filters.

**Before running:** copy `.env.example` to `.env` and fill in your FreeSQL credentials and DSN.

## 1. Connect to Oracle AI Database

`load_dotenv()` reads your `.env` file, then we connect using `python-oracledb`

In [1]:
import os
from dotenv import load_dotenv
import oracledb

load_dotenv()

connection = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASSWORD"],
    dsn=os.environ["ORACLE_DSN"],
)
print("Connected:", connection.version)

Connected: 23.9.0.25.7


## 2. Pull recent issues from GitHub

Public REST API, no auth needed for this small a request. 60 calls per hour is the unauthenticated limit

In [2]:
import requests

url = "https://api.github.com/repos/oracle/python-oracledb/issues"
params = {"state": "all", "per_page": 15}
issues = requests.get(url, params=params).json()
print(f"Pulled {len(issues)} issues")

# Peek at the first one
print(f"\nExample: #{issues[0]['number']} [{issues[0]['state']}] {issues[0]['title']}")

Pulled 15 issues

Example: #594 [open] t-strings for parameterized quries


## 3. Shape issues into LangChain Documents

Title plus body becomes the searchable content. State, labels, and issue number become metadata for filtering later.

In [3]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content=f"{i['title']}\n\n{i.get('body') or ''}",
        metadata={
            "number": i["number"],
            "state": i["state"],
            "labels": ",".join(l["name"] for l in i["labels"]),
            "url": i["html_url"],
        },
    )
    for i in issues
]

print(f"Built {len(docs)} LangChain documents")

Built 15 LangChain documents


## 4. Load the embedding model

`all-MiniLM-L6-v2` is small (~90MB), fast on CPU, and produces 384-dimensional vectors. The devcontainer pre-cached this during build, so this cell runs in a second or two.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model ready")

/home/vscode/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:01<00:00, 56.70it/s]


Embedding model ready


## 5. Store as vectors in Oracle

`OracleVS.from_documents()` does three things in one call: creates the table with a `VECTOR(384, FLOAT32)` column, generates embeddings for every document, and inserts them all.

In [5]:
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy

# Drop the table if it exists so re-runs start clean
try:
    cursor = connection.cursor()
    cursor.execute("DROP TABLE gh_issues PURGE")
    connection.commit()
except oracledb.DatabaseError:
    pass  # Table didn't exist, fine

vs = OracleVS.from_documents(
    documents=docs,
    embedding=embeddings,
    client=connection,
    table_name="GH_ISSUES",
    distance_strategy=DistanceStrategy.COSINE,
)
print("Issues stored as vectors")

Issues stored as vectors


## 6. Run a similarity search

Ask for the 5 issues semantically closest to "connection pool errors." Results come back ranked by cosine distance.

In [6]:
results = vs.similarity_search("connection pool errors", k=5)
for r in results:
    title = r.page_content.split("\n")[0]
    print(f"#{r.metadata['number']} [{r.metadata['state']}] {title}")

#576 [closed] Connection failing while connecting with create_pool_async but it works with connect function
#581 [closed] Thin mode: pool grows beyond max after TIMEDWAIT requests time out
#587 [closed] Thin async crashes with AttributeError: 'NoneType' object has no attribute 'decode' in _process_keyword_value_pairs
#579 [open] DPY-6005 / ORA-12506: All documented Thin mode IAM token approaches fail on ADB-S Private Endpoint — TOKEN_AUTH not included in TNS Connect packet
#589 [closed] IFILE path incorrectly appended to CONFIG_DIR


## 7. Compare to keyword search

Same logical question, but as a plain SQL `LIKE` query.

In [7]:
cursor = connection.cursor()
cursor.execute("""
    SELECT id, text
    FROM gh_issues
    WHERE LOWER(text) LIKE '%connection pool errors%'
    FETCH FIRST 5 ROWS ONLY
""")
rows = cursor.fetchall()
print(f"Keyword matches: {len(rows)}")
for row in rows:
    print(row)

Keyword matches: 0


## 8. Hybrid filter: semantic + metadata

Same semantic query, restricted to issues where `state = 'open'`. The filter pushes down into the SQL alongside the vector distance calculation.

In [8]:
results = vs.similarity_search(
    "connection pool errors",
    k=5,
    filter={"state": "open"},
)
for r in results:
    print(f"#{r.metadata['number']} [{r.metadata['state']}] {r.page_content[:80]}")

#579 [open] DPY-6005 / ORA-12506: All documented Thin mode IAM token approaches fail on ADB-
#592 [open] Thin mode fails with ORA-24964 when trying to open a PDB migrated from Oracle 19
#593 [open] Thin mode direct path load fails when arrow type is boolean and db type is NUM_B
#594 [open] t-strings for parameterized quries

Are there plans to add support for using t-s


## 9. Behind the abstraction: the raw SQL

Everything `similarity_search` does is a wrapper around Oracle's native vector SQL. Here's the same query as cell 6, written as pure SQL. `VECTOR_DISTANCE` is the function, `COSINE` is the distance strategy, `ORDER BY distance` gives you nearest first, `FETCH FIRST N ROWS ONLY` is your top-k.

In [9]:
import array

query_vec = embeddings.embed_query("connection pool errors")
query_vec = array.array("f", query_vec)

cursor = connection.cursor()
cursor.execute("""
    SELECT JSON_VALUE(metadata, '$.number') AS issue_number,
           JSON_VALUE(metadata, '$.state') AS state,
           VECTOR_DISTANCE(embedding, :qv, COSINE) AS distance
    FROM gh_issues
    ORDER BY distance
    FETCH FIRST 5 ROWS ONLY
""", qv=query_vec)

for issue_number, state, distance in cursor.fetchall():
    print(f"#{issue_number} [{state}]  distance={distance:.3f}")

#576 [closed]  distance=0.402
#581 [closed]  distance=0.438
#587 [closed]  distance=0.634
#579 [open]  distance=0.671
#589 [closed]  distance=0.671


## 10. Cleanup

Skip this cell if you want to keep the data around to experiment with.

In [10]:
cursor = connection.cursor()
cursor.execute("DROP TABLE gh_issues PURGE")
connection.commit()
print("Cleaned up")

Cleaned up
